In [1]:
from pyspark.sql.functions import col, from_unixtime, explode

# 1. Define Paths
storage_account = "stearthquakedata"
bronze_path = f"abfss://datalake@{storage_account}.dfs.core.windows.net/1-bronze/*.json"
silver_path = f"abfss://datalake@{storage_account}.dfs.core.windows.net/2-silver/turkey_quakes_clean"

# 2. Read the Raw JSON
# USGS JSON has a 'features' array that contains all earthquake details
df_raw = spark.read.option("multiline", "true").json(bronze_path)

# 3. Flatten the Nested Structure
# 'explode' the features array to get one row per earthquake
df_flattened = df_raw.select(explode("features").alias("quake")) \
    .select(
        col("quake.properties.mag").alias("magnitude"),
        col("quake.properties.place").alias("location"),
        # Convert milliseconds (epoch) to a readable Timestamp
        from_unixtime(col("quake.properties.time") / 1000).cast("timestamp").alias("event_time"),
        col("quake.geometry.coordinates")[0].alias("longitude"),
        col("quake.geometry.coordinates")[1].alias("latitude"),
        col("quake.properties.type").alias("event_type")
    )

# 4. Filter data
df_clean = df_flattened.filter(col("magnitude") >= 2.5)


from pyspark.sql.functions import split, trim, when, to_date

# 5. Extract location name from '20 km S of Refahiye, Turkey'
# Split by ' of ', take the second part, then split by ',' and take the first part.
df_clean = df_clean.withColumn("city", 
    when(col("location").contains(" of "), 
         trim(split(split(col("location"), " of ")[1], ",")[0]))
    .otherwise(trim(split(col("location"), ",")[0]))
)

# 6. Add a clean 'Date' column for Power BI
df_clean = df_clean.withColumn("event_date", to_date(col("event_time")))

# 7. Final columns for Silver
df_clean = df_clean.select("event_date", "event_time", "magnitude", "city", "location", "longitude", "latitude")



# 8. Write to Silver as Parquet

df_clean.write.mode("overwrite").parquet(silver_path)

print("Bronze data flattened and saved to Silver.")

StatementMeta(aspEarth, 1, 2, Finished, Available, Finished)

Success! Bronze data flattened and saved to Silver.


In [2]:
%%sql
/* SQL to query to Silver data */
SELECT 
    city, 
    magnitude, 
    event_date 
FROM 
    parquet.`abfss://datalake@stearthquakedata.dfs.core.windows.net/2-silver/turkey_quakes_clean`
ORDER BY 
    magnitude DESC
LIMIT 10

StatementMeta(aspEarth, 1, 3, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 3 fields>